# CS769 - Experiment 4 (Kaggle): DPO→merge vs merge→DPO at equal DPO budget

**Settings:** Accelerator = *GPU T4 x2*, Internet = *On*. Add your repo as a Kaggle Dataset (see cell 3).

| | Pipeline | DPO data | Steps |
|---|---|---|---|
| W1 | SFT experts → merge | none | 0 |
| W2 | each SFT expert → DPO (own shard) → merge | 664/664/672 disjoint, token-balanced pairs | 83+83+84 = 250 |
| W3 | SFT experts → merge → DPO | full 2,000-pair pool | 250 |

Merges: Linear, TIES (density 0.2), DARE-TIES (drop 0.5, density 0.2, seed 10), equal weights 1/3. Every job is resumable: if the session dies, re-run the same cells and finished jobs are skipped.

In [ ]:
!nvidia-smi -L
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!pip install -q -U peft datasets accelerate
import transformers, peft; print(transformers.__version__, peft.__version__)

In [ ]:
# 3. Copy the repo into the working dir.  EDIT DATASET_DIR to your Kaggle dataset path.
# The dataset must contain (at its root) the CS769 layout:
#   configs/  src/ (incl. src/e4/e4_pipeline.py, src/eval/*, src/train/train_dpo_stable.py, src/data/precompute_ref_logps.py)
#   outputs/adapters/{e_math,e_code,e_instr,e_pref}   outputs/datasets/{e_pref_train,e_pref_test}.jsonl
#   outputs/results/b0_pref_200_cache.json   (optional, only for a reproducibility check)
import os, shutil, glob
DATASET_DIR = "/kaggle/input/cs769-repo"      # <-- edit
REPO = "/kaggle/working/CS769"
if not os.path.exists(REPO):
    shutil.copytree(DATASET_DIR, REPO)
os.chdir(REPO)
need = ["src/e4/e4_pipeline.py","src/eval/eval_math.py","src/eval/eval_code.py","src/eval/eval_ifeval.py",
        "src/train/train_dpo_stable.py","src/data/precompute_ref_logps.py","configs/base_config.py",
        "outputs/datasets/e_pref_train.jsonl","outputs/datasets/e_pref_test.jsonl"] + \
       [f"outputs/adapters/e_{n}/adapter_config.json" for n in ["math","code","instr","pref"]]
missing = [p for p in need if not os.path.exists(p)]
print("missing:", missing or "none")
print("train lines:", sum(1 for _ in open("outputs/datasets/e_pref_train.jsonl")),
      "| test lines:", sum(1 for _ in open("outputs/datasets/e_pref_test.jsonl")))

In [ ]:
# 4. Job runner: one subprocess per job, pinned to a free GPU, 2 jobs in parallel on T4 x2.
import subprocess, sys, os, queue, concurrent.futures as cf
N_GPU = max(1, len(subprocess.run("nvidia-smi -L", shell=True, capture_output=True, text=True).stdout.strip().splitlines()))
print("GPUs:", N_GPU)
def _run(job, gpu, extra):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu))
    os.makedirs("outputs/e4/logs", exist_ok=True)
    with open(f"outputs/e4/logs/{job}.log", "a") as lg:
        return subprocess.run([sys.executable, "src/e4/e4_pipeline.py", "run", "--job", job] + extra,
                              env=env, stdout=lg, stderr=subprocess.STDOUT).returncode
def run_queue(jobs, extra=()):
    gpus = queue.Queue()
    for g in range(N_GPU): gpus.put(g)
    def worker(job):
        g = gpus.get()
        try: rc = _run(job, g, list(extra))
        finally: gpus.put(g)
        print(("OK   " if rc == 0 else "FAIL ") + job, flush=True)
        return job, rc
    with cf.ThreadPoolExecutor(N_GPU) as ex: res = list(ex.map(worker, jobs))
    failed = [j for j, rc in res if rc]
    for j in failed:
        print("\n--- log tail:", j); print(open(f"outputs/e4/logs/{j}.log").read()[-2000:])
    return failed
def status(): print(subprocess.run([sys.executable, "src/e4/e4_pipeline.py", "status"], capture_output=True, text=True).stdout)

## Stage 0 - shards, bake SFT adapters, B0 reference cache, W1 merges

In [ ]:
f1 = run_queue(["shards", "bake_sft_math", "bake_sft_code", "bake_sft_instr", "bake_sft_pref", "b0_ref500"])
assert not f1, f1
f2 = run_queue([f"merge_w1_{m}" for m in ["linear", "ties", "dare_ties"]])
assert not f2, f2
status()


## Smoke test (3 DPO steps) - run this BEFORE the long stages; it prints s/step and a GPU-hour estimate

In [ ]:
print(subprocess.run("CUDA_VISIBLE_DEVICES=0 python src/e4/e4_pipeline.py run --job bench 2>&1 | tail -n 15", shell=True, capture_output=True, text=True).stdout)

## Stage 1 - the 6 DPO jobs (W2 building blocks first, then W3 per merge method)

In [ ]:
fails = run_queue(["dpo_exp_math", "dpo_exp_code", "dpo_exp_instr",
                   "dpo_w3_linear", "dpo_w3_ties", "dpo_w3_dare_ties"])
assert not fails, fails

## Stage 2 - W2 merges (merge of the three DPO-aligned experts)

In [ ]:
fails = run_queue([f"merge_w2_{m}" for m in ["linear", "ties", "dare_ties"]])
assert not fails, fails

## Stage 3 - evaluation (SVAMP 300, MBPP 257, IFEval 300 strict prompt-level, UltraFeedback 500 pairs)
`--delete_after` frees disk once no later job needs a checkpoint. Anchors (B0, SFT experts, E_pref) are re-evaluated on this hardware so CR/PRR use consistent denominators.

In [ ]:
tags = ["b0", "e_math", "e_code", "e_instr", "e_pref", "al_math", "al_code", "al_instr"] + \
       [f"{w}_{m}" for w in ("w1", "w2", "w3") for m in ("linear", "ties", "dare_ties")]
fails = run_queue([f"eval_{t}" for t in tags], extra=["--delete_after"])
status()

## Stage 4 - report (Table E4, PRR with CIs, W2−W3 paired comparison) and download pack

In [ ]:
print(subprocess.run([sys.executable, "src/e4/e4_pipeline.py", "run", "--job", "report"], capture_output=True, text=True).stdout)
print(subprocess.run([sys.executable, "src/e4/e4_pipeline.py", "run", "--job", "pack"], capture_output=True, text=True).stdout)

In [ ]:
# Per-job DPO accounting (pairs, steps, tokens, runtime) straight from the metrics files
import json, glob
for f in sorted(glob.glob("outputs/e4/train_metrics/*.json")):
    m = json.load(open(f))
    print(f"{m['tag']:<14} pairs={m['pairs']:<5} steps={m['optimizer_steps']:<4} tokens={m['tokens']:>9,} "
          f"train={m['train_seconds']/60:6.1f} min ({m['sec_per_step']:.1f}s/step) gpu={m['gpu']}")
from IPython.display import FileLink; display(FileLink("e4_artifacts.zip"))